# Image Restoration AI Pipeline — Interactive Demonstration

This notebook demonstrates the end-to-end automated image restoration system. The pipeline automatically detects degradations (haze, low-light, rain) using a MobileNetV2 CNN classifier, routes the image dynamically, and restores it using Transformer neural networks.

In [ ]:
import os
import sys
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io

# Ensure project root is in path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.core.pipeline import RestorationPipeline
from src.core.config import get_config
from src.utils.image import load_image

# Initialize pipeline
config = get_config()
pipeline = RestorationPipeline()
print(f"Pipeline ready! Device: {config.pipeline.device}, Detector: {config.router.detector_type}")

## Automated Pipeline Demonstration
Pass any image path to the pipeline — it will auto-detect degradations, route to the required models, and restore the image.

In [ ]:
def run_restoration_demo(image_path, reference_path=None):
    if not os.path.exists(image_path):
        print(f"Error: File not found: {image_path}")
        return

    temp_output_path = "temp_restored_demo.png"
    original_np = load_image(image_path)

    # Run restoration pipeline
    result = pipeline.process_image(
        image_path,
        save_path=temp_output_path,
        reference_path=reference_path if (reference_path and os.path.exists(reference_path)) else None
    )

    # Load restored image
    restored_np = load_image(temp_output_path) if os.path.exists(temp_output_path) else original_np

    # Plot results
    num_cols = 3 if (reference_path and os.path.exists(reference_path)) else 2
    fig, axes = plt.subplots(1, num_cols, figsize=(5 * num_cols, 5))

    axes[0].imshow(original_np)
    axes[0].set_title("Degraded Input")
    axes[0].axis("off")

    axes[1].imshow(restored_np)
    axes[1].set_title(f"Restored (Plan: {result['plan']})")
    axes[1].axis("off")

    if num_cols == 3:
        ref_np = load_image(reference_path)
        axes[2].imshow(ref_np)
        axes[2].set_title("Ground-Truth Reference")
        axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    # Print details
    print(f"Latency: {result['latency_ms']:.2f} ms")
    print("Degradation Confidence Scores:")
    for k, v in result["scores"].items():
        print(f"  {k}: {v:.4f}")
    
    print("Quality Metrics:")
    has_fr_metrics = False
    for k, v in result["metrics"].items():
        if v is not None:
            val_str = f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
            if k in ["psnr", "ssim", "lpips"]:
                has_fr_metrics = True
                print(f"  {k.upper()}: {val_str}")
            elif k == "brisque":
                print(f"  BRISQUE (No-Reference Quality): {val_str}")

    if not has_fr_metrics:
        print("  Note: PSNR, SSIM, and LPIPS require a ground-truth reference image.")
        print("  For standalone images, BRISQUE measures perceptual quality (lower score = better quality).")

    # Clean temp output
    if os.path.exists(temp_output_path):
        os.remove(temp_output_path)

### Test Pipeline with Sample Dataset Images

In [ ]:
# Test Haze Image
sample_haze = os.path.join(project_root, "images", "haze_test", "hazy", "0001_0.8_0.2.jpg")
sample_clear = os.path.join(project_root, "images", "haze_test", "clear", "0001.png")
print("=== Testing Haze Restoration ===")
run_restoration_demo(sample_haze, reference_path=sample_clear)

In [ ]:
# Test Low-Light Image
sample_low = os.path.join(project_root, "images", "lowlight_test", "Low", "low00690.png")
sample_normal = os.path.join(project_root, "images", "lowlight_test", "Normal", "normal00690.png")
print("=== Testing Low-Light Restoration ===")
run_restoration_demo(sample_low, reference_path=sample_normal)

## Interactive File Uploader
Upload any custom image from your computer, then click **Restore Image** to run the restoration pipeline.

In [ ]:
uploader = widgets.FileUpload(accept='image/*', multiple=False)
restore_btn = widgets.Button(description="Restore Image", button_style='primary', icon='magic')
output_area = widgets.Output()

def on_restore_click(b):
    output_area.clear_output(wait=True)
    with output_area:
        if not uploader.value:
            print("Please select an image file first using the Upload button.")
            return
        
        # Handle ipywidgets 7.x and 8.x formats safely
        try:
            if isinstance(uploader.value, dict):
                fname = list(uploader.value.keys())[0]
                content = uploader.value[fname]['content']
            elif isinstance(uploader.value, (tuple, list)) and len(uploader.value) > 0:
                item = uploader.value[0]
                fname = item.name if hasattr(item, 'name') else item.get('name', 'upload.png')
                content = item.content.tobytes() if hasattr(item.content, 'tobytes') else item.get('content', b'')
            else:
                print("No file content detected.")
                return
        except Exception as e:
            print(f"Upload parsing error: {e}")
            return

        temp_in = f"temp_upload_{fname}"
        temp_out = f"temp_restored_{fname}"
        
        try:
            with open(temp_in, "wb") as f:
                f.write(content)
            
            original_np = load_image(temp_in)
            result = pipeline.process_image(temp_in, save_path=temp_out)
            restored_np = load_image(temp_out) if os.path.exists(temp_out) else original_np
            
            fig, axes = plt.subplots(1, 2, figsize=(10, 5))
            axes[0].imshow(original_np)
            axes[0].set_title("Uploaded Original")
            axes[0].axis("off")
            
            axes[1].imshow(restored_np)
            axes[1].set_title(f"Restored (Plan: {result['plan']})")
            axes[1].axis("off")
            
            plt.tight_layout()
            plt.show()
            
            print(f"Latency: {result['latency_ms']:.2f} ms")
            print("Degradation Confidence Scores:")
            for k, v in result["scores"].items():
                print(f"  {k}: {v:.4f}")
                
            print("Quality Metrics:")
            has_fr_metrics = False
            for k, v in result["metrics"].items():
                if v is not None:
                    val_str = f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
                    if k in ["psnr", "ssim", "lpips"]:
                        has_fr_metrics = True
                        print(f"  {k.upper()}: {val_str}")
                    elif k == "brisque":
                        print(f"  BRISQUE (No-Reference Quality): {val_str}")

            if not has_fr_metrics:
                print("  Note: PSNR, SSIM, and LPIPS require a ground-truth reference image.")
                print("  For standalone uploads, BRISQUE measures perceptual quality (lower score = better quality).")
                
        finally:
            for p in [temp_in, temp_out]:
                if os.path.exists(p):
                    os.remove(p)

restore_btn.on_click(on_restore_click)
display(widgets.VBox([
    widgets.HTML("<h3>Interactive File Restoration:</h3>"),
    widgets.HBox([uploader, restore_btn]),
    output_area
]))